# Colab Local LLM Classifier
Runs the classifier stage locally on Colab through a vLLM OpenAI-compatible server and writes classifier-only prediction parquet files to Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/data/llm-gatekeeping'
REPO_URL = 'https://github.com/noamdwc/llm-gatekeeping.git'
REPO_DIR = '/content/llm-gatekeeping'
BRANCH = 'zero_shot_nlp_attack'

SPLIT = 'val'
LIMIT = 5  # Set to None for the full split.
MODEL_ID = 'meta/llama-3.1-8b-instruct'
TENSOR_PARALLEL_SIZE = 1
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 4096
BATCH_SIZE = 1
CHECKPOINT_EVERY = 50
OUTPUT_SUFFIX = 'colab_local_classifier'

HF_CACHE_DIR = f'{DRIVE_ROOT}/cache/huggingface'
SPLITS_DIR = f'{DRIVE_ROOT}/data/processed/splits'
PREDICTIONS_DIR = f'{DRIVE_ROOT}/data/processed/predictions'
CHECKPOINT_PATH = f'{PREDICTIONS_DIR}/llm_checkpoint_{SPLIT}_{OUTPUT_SUFFIX}.parquet'
OUTPUT_PATH = f'{PREDICTIONS_DIR}/llm_predictions_{SPLIT}_{OUTPUT_SUFFIX}.parquet'
VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'

import os

os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_HUB_CACHE'] = f'{HF_CACHE_DIR}/hub'
os.environ['TRANSFORMERS_CACHE'] = f'{HF_CACHE_DIR}/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{HF_CACHE_DIR}/datasets'
print(f'Hugging Face cache: {HF_CACHE_DIR}')
print(f'Output path: {OUTPUT_PATH}')

In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Using existing repo at {REPO_DIR}')

os.chdir(REPO_DIR)
print('Repo:', os.getcwd())

subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q "vllm>=0.6.0" "openai>=1.0.0"